# Data cleaner for JupyterLite (2025)

This code helps you clean social media data for text analysis. As you can tell from the variable names below, the original code was written for working with Twitter content. However, it will work with any other text upload you provide in `.txt` format.

*JupyterLite* runs Python directly in your browser. When you first open this notebook, the Python kernel may take several seconds to start. Wait until the kernel is ready before running the cells. You will see an empty white circle next to "Python (Pyodide)" in the top right corner of the kernel when you are good to go.

To provide a file:

1. Upload your `.txt` file using the **Upload** button in the JupyterLite file browser on the left.
2. Return to this notebook.
3. Run the cells below.

The cleaned file will be provided as a download at the end.

Run the cells below and follow the instructions in the console output. The first cell installs the Python packages needed inside the browser (this can take a few seconds the first time).

In [1]:
import re
import base64
from pathlib import Path
from IPython.display import HTML, display

print("Import complete!")

Import complete!


The second step is to define the cleaning operations, including a list of stopwords. You will be able to edit those stopwords in a dialogue box when running the cleaning function, so please do not change them in the cell below.

In [2]:
# Default stopwords
DEFAULT_STOPWORDS = {
    "a", "an", "the", "is", "are", "was", "were", "be", "been", "being",
    "and", "or", "but", "if", "then", "else", "when", "while",
    "for", "to", "of", "in", "on", "at", "by", "with", "from",
    "this", "that", "these", "those", "it", "its", "as", "not", "no",
    "so", "too", "very",

    # social media / URL elements
    "http", "https", "www", "com", "co", "org"
}


def clean_text(text, stopwords):
    """Clean one line of text and return space-separated tokens."""

    if text is None:
        return ""

    if not isinstance(text, str):
        text = str(text)

    # Lowercase
    text = text.lower()

    # Remove apostrophes
    text = re.sub(r"['’]", "", text)

    # Remove mentions
    text = re.sub(r"@[A-Za-z0-9_]+", " ", text)

    # Remove URLs
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    # Remove bracketed text
    text = re.sub(r"\[.*?\]", " ", text)

    # Replace punctuation with spaces.
    # \w is Unicode-aware, so letters such as é, ü, č, etc. are preserved.
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)

    # Remove underscores
    text = re.sub(r"_+", " ", text)

    # Tokenize and remove stopwords
    tokens = [
        word for word in text.split()
        if word and word not in stopwords
    ]

    return " ".join(tokens)

In the third step, you can upload your data in .txt format and apply the stopwords and cleaning function. Do not close the browser while the script runs, and do not forget to download your cleaned data at the end.

In [ ]:
# Find .txt files in the JupyterLite working directory
txt_files = sorted(Path(".").glob("*.txt"))

if not txt_files:
    print("No .txt files found.")
    print("Please upload a .txt file using the JupyterLite file browser,")
    print("then run this cell again.")

else:
    print("Available text files:")

    # Create dictionary from available txt files
    file_dict = {
        i: path
        for i, path in enumerate(txt_files, start=1)
    }

    # Display available files
    for i, path in file_dict.items():
        print(f"  {i}: {path.name}")

    print()

    # Get the user's choice.
    # In JupyterLite, input() needs to be awaited.
    choice = await input(
        f"Enter the number of the file you want to clean (1-{len(file_dict)}): "
    )

    try:
        choice = int(choice)
        input_file = file_dict[choice]
    except (ValueError, KeyError):
        raise ValueError("Invalid file selection.")

    print(f"Selected: {input_file.name}")

    # Allows you to edit the stopword list
    default_text = " ".join(sorted(DEFAULT_STOPWORDS))

    print()
    print("Default stopwords:")
    print(default_text)
    print()
    print("Press Enter to use the default list.")
    print("Otherwise, enter your own space-separated stopwords.")

    # In JupyterLite, input() needs to be awaited here as well.
    custom_stopwords = (await input("Stopwords: ")).strip()

    if custom_stopwords:
        stopwords = set(custom_stopwords.split())
    else:
        stopwords = DEFAULT_STOPWORDS.copy()

    print(f"{len(stopwords)} stopwords will be removed.")

You can now start the cleaning operations!


In [ ]:
# Read the input file
try:
    text = input_file.read_text(encoding="utf-8")
except UnicodeDecodeError:
    print("UTF-8 decoding failed. Trying Latin-1...")
    text = input_file.read_text(encoding="latin-1")

lines = text.splitlines()

print(f"Input file: {input_file.name}")
print(f"Number of lines: {len(lines)}")

# Clean each line
cleaned = [clean_text(line, stopwords) for line in lines]

# Preview
preview_n = min(10, len(cleaned))

print()
print(f"Preview of first {preview_n} cleaned lines:")
print("-" * 60)

for i in range(preview_n):
    print(cleaned[i])

We will now create the output text. Once your data download is complete, you can close the browser or move on to another script.

In [ ]:
# Create output text
cleaned_text = "\n".join(cleaned) + "\n"

# Create filename
output_filename = input_file.stem + "_cleaned.txt"

# Create browser download link
encoded = base64.b64encode(
    cleaned_text.encode("utf-8")
).decode("ascii")

href = f"data:text/plain;base64,{encoded}"

display(HTML(
    f'''
    <p><b>Download your cleaned file:</b></p>
    <a download="{output_filename}" href="{href}">
        {output_filename}
    </a>
    '''
))